# Cyclistic — Consolidación y preparación de datos

## Objetivo

Este notebook corresponde a la etapa de preparación y limpieza de datos del caso de estudio Cyclistic, desarrollado como trabajo final del Google Data Analytics Professional Certificate.

A partir de los controles exploratorios realizados anteriormente, se busca consolidar los registros de los 12 meses en un único conjunto de datos, verificar su consistencia y prepararlo para el análisis de los patrones de uso de las bicicletas.

## Datos utilizados

Se utilizan los archivos mensuales correspondientes al período de septiembre de 2025 a agosto de 2026.

Los archivos se procesan utilizando Python y Pandas en Google Colab, con los datos almacenados en Google Drive.

**Entorno de trabajo:** este notebook fue desarrollado en Google Colab utilizando archivos almacenados en Google Drive. Las rutas corresponden al entorno original de trabajo y deben adaptarse para ejecutar el análisis en otro entorno.

## Procedimientos realizados

- Consolidación de los 12 archivos mensuales.
- Identificación y tratamiento de registros duplicados.
- Revisión de fechas y duraciones de los viajes, incluyendo posibles anomalías asociadas a cambios de horario.
- Identificación de valores extremos mediante indicadores, sin eliminarlos automáticamente.
- Preparación y exportación del conjunto de datos para las siguientes etapas del análisis.

**Propósito:** obtener un conjunto de datos consistente y documentar las decisiones de preparación que pueden influir en los resultados.


In [ ]:
import pandas as pd
import glob
import os

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
carpeta_originales = "/content/drive/MyDrive/Colab Notebooks/cyclistic/datos_originales"

ruta = os.path.join(carpeta_originales, "*.csv")

archivos = sorted(glob.glob(ruta))

print("Archivos encontrados:", len(archivos))

for archivo in archivos:
    print(os.path.basename(archivo))

Archivos encontrados: 12
202509-divvy-tripdata.csv
202510-divvy-tripdata.csv
202511-divvy-tripdata.csv
202512-divvy-tripdata.csv
202601-divvy-tripdata.csv
202602-divvy-tripdata.csv
202603-divvy-tripdata.csv
202604-divvy-tripdata.csv
202605-divvy-tripdata.csv
202606-divvy-tripdata.csv
202607-divvy-tripdata.csv
202608-divvy-tripdata.csv


In [ ]:
resumen_archivos = []

for archivo in archivos:
    df = pd.read_csv(archivo, engine="python")

    resumen_archivos.append({
        "archivo": os.path.basename(archivo),
        "registros": len(df)
    })

resumen_archivos = pd.DataFrame(resumen_archivos)

display(resumen_archivos)

print("Total de registros:", resumen_archivos["registros"].sum())

,archivo,registros
0,202509-divvy-tripdata.csv,714759
1,202510-divvy-tripdata.csv,646039
2,202511-divvy-tripdata.csv,356628
3,202512-divvy-tripdata.csv,140534
4,202601-divvy-tripdata.csv,137787
5,202602-divvy-tripdata.csv,201450
6,202603-divvy-tripdata.csv,317037
7,202604-divvy-tripdata.csv,448252
8,202605-divvy-tripdata.csv,653704
9,202606-divvy-tripdata.csv,762550


Total de registros: 6115982


In [ ]:
datos_mensuales = []

for archivo in archivos:
    df = pd.read_csv(archivo, engine="python")

    # Convertir fechas
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df["ended_at"] = pd.to_datetime(df["ended_at"], errors="coerce")

    # Calcular duración
    duracion = df["ended_at"] - df["started_at"]

    df["ride_length_min"] = duracion.dt.total_seconds() / 60
    df["ride_length_sec"] = duracion.dt.total_seconds()

    # Guardar el mes procesado
    datos_mensuales.append(df)

    print(os.path.basename(archivo), "procesado")

202509-divvy-tripdata.csv procesado
202510-divvy-tripdata.csv procesado
202511-divvy-tripdata.csv procesado
202512-divvy-tripdata.csv procesado
202601-divvy-tripdata.csv procesado
202602-divvy-tripdata.csv procesado
202603-divvy-tripdata.csv procesado
202604-divvy-tripdata.csv procesado
202605-divvy-tripdata.csv procesado
202606-divvy-tripdata.csv procesado
202607-divvy-tripdata.csv procesado
202608-divvy-tripdata.csv procesado


In [ ]:
informe_calidad = []

for archivo, df in zip(archivos, datos_mensuales):

    informe_calidad.append({
        "mes": os.path.basename(archivo)[:6],
        "registros": len(df),
        "duracion_nula": df["ride_length_min"].isna().sum(),
        "duracion_no_positiva": (df["ride_length_min"] <= 0).sum(),
        "membresia_invalida": (~df["member_casual"].isin(["member", "casual"])).sum()
    })

informe_calidad = pd.DataFrame(informe_calidad)

display(informe_calidad)

,mes,registros,duracion_nula,duracion_no_positiva,membresia_invalida
0,202509,714759,0,0,0
1,202510,646039,0,0,0
2,202511,356628,0,29,0
3,202512,140534,0,0,0
4,202601,137787,0,0,0
5,202602,201450,0,0,0
6,202603,317037,0,0,0
7,202604,448252,0,0,0
8,202605,653704,0,0,0
9,202606,762550,0,0,0


## Revisión de anomalías asociadas a cambios de horario

Durante la preparación de los archivos mensuales se identificaron registros con duraciones negativas o inconsistentes, especialmente en fechas coincidentes a los cambios de horario de verano.

Se investigó la posibilidad de que algunas de estas anomalías estuvieran relacionadas con los ajustes de hora, que pueden afectar el cálculo de la duración cuando se utilizan fechas y horas locales.

A partir de esta revisión se realizaron ajustes sobre los registros que cumplían los criterios definidos, procurando conservar la información original siempre que fuera posible.

**Consideración:** la relación entre estas anomalías y los cambios de horario constituye una hipótesis basada en los patrones observados, no una causa confirmada para cada registro.

In [ ]:
noviembre = datos_mensuales[2]

casos_invalidos = noviembre[
    noviembre["ride_length_min"] <= 0
]

display(
    casos_invalidos[
        ["ride_id", "started_at", "ended_at",
         "ride_length_min", "member_casual"]
    ]
)

,ride_id,started_at,ended_at,ride_length_min,member_casual
25121,5D010AFEA6850513,2025-11-02 01:52:37.475,2025-11-02 01:13:24.728,-39.212450,member
26669,D1E5316AD88ECD45,2025-11-02 01:50:39.702,2025-11-02 01:14:04.164,-36.592300,casual
28663,3AF2F8908C9386F8,2025-11-02 01:57:57.512,2025-11-02 01:33:52.225,-24.088117,member
43559,19386939ECD81B33,2025-11-02 01:55:34.399,2025-11-02 01:21:31.873,-34.042100,member
51927,083534D28DA37F72,2025-11-02 01:17:57.001,2025-11-02 01:05:27.752,-12.487483,member
96512,4A659F1BB40EDBF7,2025-11-02 01:55:38.177,2025-11-02 01:04:35.315,-51.047700,casual
122065,DA5386E5759667EC,2025-11-02 01:51:31.704,2025-11-02 01:05:47.339,-45.739417,member
127523,985A3462DAACFFCF,2025-11-02 01:59:07.835,2025-11-02 01:05:19.214,-53.810350,member
147446,434E3D11ABA96891,2025-11-02 01:59:40.156,2025-11-02 01:08:51.303,-50.814217,casual
157264,9288498F5630E3AD,2025-11-02 01:49:48.360,2025-11-02 01:02:40.976,-47.123067,casual


In [ ]:
marzo = datos_mensuales[6]

inicio = pd.Timestamp("2026-03-08 00:00:00")
cambio = pd.Timestamp("2026-03-08 02:00:00")
despues = pd.Timestamp("2026-03-08 03:00:00")
fin_dia = pd.Timestamp("2026-03-09 00:00:00")

casos_marzo = marzo[
    (marzo["started_at"] >= inicio) &
    (marzo["started_at"] < cambio) &
    (marzo["ended_at"] >= despues)
]

print("Viajes potencialmente afectados:", len(casos_marzo))

display(
    casos_marzo[
        ["ride_id", "started_at", "ended_at",
         "ride_length_min", "member_casual"]
    ]
)

Viajes potencialmente afectados: 28


,ride_id,started_at,ended_at,ride_length_min,member_casual
21842,46016458BAE88688,2026-03-08 01:57:28.156,2026-03-08 03:00:09.807,62.694183,member
32959,1230AA19222F2336,2026-03-08 01:34:35.076,2026-03-08 03:15:19.007,100.732183,member
53585,1AC2DC825D274635,2026-03-08 01:59:40.948,2026-03-08 03:26:51.691,87.179050,casual
62096,264495DA8658F811,2026-03-08 01:55:36.604,2026-03-08 03:03:51.010,68.240100,member
70015,749CCC0B4E6DD5C9,2026-03-08 01:47:48.084,2026-03-08 03:29:38.266,101.836367,member
77441,13BA16BED813E5EF,2026-03-08 01:45:46.299,2026-03-08 03:08:12.003,82.428400,member
77477,7ED45F51281FD6A4,2026-03-08 01:53:34.754,2026-03-08 03:02:09.125,68.572850,casual
85018,B0256D1597A3C727,2026-03-08 01:57:48.837,2026-03-08 03:00:53.487,63.077500,member
104032,A84D638D4109E652,2026-03-08 01:59:03.112,2026-03-08 03:20:19.559,81.274117,casual
113819,947ABE63E588EBB5,2026-03-08 01:43:28.379,2026-03-08 03:07:29.927,84.025800,casual


In [ ]:
# Crear copias de los datos mensuales
datos_ajustados = [df.copy() for df in datos_mensuales]

# Inicializar columnas de auditoría
for df in datos_ajustados:
    df["ride_length_min_original"] = float("nan")
    df["ajuste_horario"] = 0

# Seleccionar marzo
marzo_ajustado = datos_ajustados[6]

# Identificar los viajes afectados
condicion_marzo = (
    (marzo_ajustado["started_at"] >= pd.Timestamp("2026-03-08 00:00:00")) &
    (marzo_ajustado["started_at"] < pd.Timestamp("2026-03-08 02:00:00")) &
    (marzo_ajustado["ended_at"] >= pd.Timestamp("2026-03-08 03:00:00"))
)

# Guardar duración original
marzo_ajustado.loc[
    condicion_marzo, "ride_length_min_original"
] = marzo_ajustado.loc[condicion_marzo, "ride_length_min"]

# Registrar ajuste
marzo_ajustado.loc[condicion_marzo, "ajuste_horario"] = -60

# Corregir duración en la columna existente
marzo_ajustado.loc[condicion_marzo, "ride_length_min"] -= 60

# Verificar
display(
    marzo_ajustado.loc[
        condicion_marzo,
        ["ride_id", "ride_length_min_original",
         "ajuste_horario", "ride_length_min"]
    ]
)

,ride_id,ride_length_min_original,ajuste_horario,ride_length_min
21842,46016458BAE88688,62.694183,-60,2.694183
32959,1230AA19222F2336,100.732183,-60,40.732183
53585,1AC2DC825D274635,87.179050,-60,27.179050
62096,264495DA8658F811,68.240100,-60,8.240100
70015,749CCC0B4E6DD5C9,101.836367,-60,41.836367
77441,13BA16BED813E5EF,82.428400,-60,22.428400
77477,7ED45F51281FD6A4,68.572850,-60,8.572850
85018,B0256D1597A3C727,63.077500,-60,3.077500
104032,A84D638D4109E652,81.274117,-60,21.274117
113819,947ABE63E588EBB5,84.025800,-60,24.025800


In [ ]:
inicio_hora = pd.Timestamp("2026-03-08 02:00:00")
fin_hora = pd.Timestamp("2026-03-08 03:00:00")

inicios_invalidos = (
    (marzo_ajustado["started_at"] >= inicio_hora) &
    (marzo_ajustado["started_at"] < fin_hora)
).sum()

finales_invalidos = (
    (marzo_ajustado["ended_at"] >= inicio_hora) &
    (marzo_ajustado["ended_at"] < fin_hora)
).sum()

print("Viajes iniciados durante la hora inexistente:", inicios_invalidos)
print("Viajes finalizados durante la hora inexistente:", finales_invalidos)

Viajes iniciados durante la hora inexistente: 0
Viajes finalizados durante la hora inexistente: 0


In [ ]:
noviembre = datos_ajustados[2]

inicio = pd.Timestamp("2025-11-02 01:00:00")
fin = pd.Timestamp("2025-11-02 02:00:00")

casos_ambiguos = noviembre[
    (noviembre["started_at"] >= inicio) &
    (noviembre["started_at"] < fin) &
    (noviembre["ended_at"] >= inicio) &
    (noviembre["ended_at"] < fin)
]

print("Viajes dentro de la hora repetida:", len(casos_ambiguos))
print(
    "Con duración negativa:",
    (casos_ambiguos["ride_length_min"] < 0).sum()
)
print(
    "Con duración positiva:",
    (casos_ambiguos["ride_length_min"] > 0).sum()
)

Viajes dentro de la hora repetida: 391
Con duración negativa: 29
Con duración positiva: 362


In [ ]:
noviembre_ajustado = datos_ajustados[2]

In [ ]:
noviembre_ajustado["horario_ambiguo"] = 0

condicion_ambigua = (
    (noviembre_ajustado["started_at"] >= pd.Timestamp("2025-11-02 01:00:00")) &
    (noviembre_ajustado["started_at"] < pd.Timestamp("2025-11-02 02:00:00")) &
    (noviembre_ajustado["ended_at"] >= pd.Timestamp("2025-11-02 01:00:00")) &
    (noviembre_ajustado["ended_at"] < pd.Timestamp("2025-11-02 02:00:00")) &
    (noviembre_ajustado["ajuste_horario"] == 0)
)

noviembre_ajustado.loc[condicion_ambigua, "horario_ambiguo"] = 1

print("Viajes con horario ambiguo:", condicion_ambigua.sum())

Viajes con horario ambiguo: 391


In [ ]:
print(noviembre_ajustado["ajuste_horario"].value_counts())

ajuste_horario
0    356628
Name: count, dtype: int64


In [ ]:
# Identificar duraciones negativas
condicion_noviembre = noviembre_ajustado["ride_length_min"] < 0

# Guardar la duración original
noviembre_ajustado.loc[
    condicion_noviembre, "ride_length_min_original"
] = noviembre_ajustado.loc[
    condicion_noviembre, "ride_length_min"
]

# Registrar el ajuste
noviembre_ajustado.loc[condicion_noviembre, "ajuste_horario"] = 60

# Corregir la duración
noviembre_ajustado.loc[condicion_noviembre, "ride_length_min"] += 60

# Actualizar duración en segundos
noviembre_ajustado.loc[condicion_noviembre, "ride_length_sec"] = (
    noviembre_ajustado.loc[condicion_noviembre, "ride_length_min"] * 60
)

print("Viajes corregidos:", condicion_noviembre.sum())
print(noviembre_ajustado["ajuste_horario"].value_counts())

Viajes corregidos: 29
ajuste_horario
0     356599
60        29
Name: count, dtype: int64


In [ ]:
noviembre_ajustado["horario_ambiguo"] = 0

noviembre_ajustado.loc[
    condicion_ambigua &
    (noviembre_ajustado["ajuste_horario"] == 0),
    "horario_ambiguo"
] = 1

print(noviembre_ajustado["horario_ambiguo"].value_counts())

horario_ambiguo
0    356266
1       362
Name: count, dtype: int64


## Identificación de valores extremos

A partir de los criterios definidos en la exploración inicial, se identifican como valores extremos los viajes con una duración inferior a 1 minuto o superior a 120 minutos.

Se incorpora la variable `extremo`, que permite identificar estos registros sin eliminarlos del conjunto de datos.

Esta decisión busca conservar la información disponible y permitir que, durante el análisis, se puedan comparar los resultados incluyendo o excluyendo los valores extremos.

In [ ]:
for df in datos_ajustados:
    df["extremo"] = (
        (df["ride_length_min"] < 1) |
        (df["ride_length_min"] > 120)
    ).astype(int)

total_extremos = sum(df["extremo"].sum() for df in datos_ajustados)
total_viajes = sum(len(df) for df in datos_ajustados)

print("Viajes extremos:", total_extremos)
print("Porcentaje:", round(total_extremos / total_viajes * 100, 2), "%")

Viajes extremos: 188918
Porcentaje: 3.09 %


In [ ]:
for df in datos_ajustados:
    if "horario_ambiguo" not in df.columns:
        df["horario_ambiguo"] = 0

print("Columnas uniformes:",
      all(
          list(df.columns) == list(datos_ajustados[0].columns)
          for df in datos_ajustados
      ))

Columnas uniformes: False


In [ ]:
columnas_referencia = set(datos_ajustados[0].columns)

for i, df in enumerate(datos_ajustados):
    if set(df.columns) != columnas_referencia:
        print("Diferencia en:", os.path.basename(archivos[i]))
        print("Columnas faltantes:", columnas_referencia - set(df.columns))
        print("Columnas adicionales:", set(df.columns) - columnas_referencia)

print(
    "¿Todos tienen las mismas columnas?",
    all(set(df.columns) == columnas_referencia for df in datos_ajustados)
)

¿Todos tienen las mismas columnas? True


In [ ]:
columnas_referencia = datos_ajustados[0].columns.tolist()

datos_ajustados = [
    df[columnas_referencia].copy()
    for df in datos_ajustados
]

print(
    "Columnas uniformes:",
    all(
        list(df.columns) == columnas_referencia
        for df in datos_ajustados
    )
)

Columnas uniformes: True


## Consolidación de los archivos mensuales

Los registros de viajes se encuentran distribuidos en 12 archivos CSV, uno por cada mes del período de estudio.

Para realizar un análisis conjunto, se cargan los archivos utilizando Pandas y se concatenan en un único DataFrame.

Esta consolidación permite trabajar con todo el período y facilita las posteriores tareas de limpieza, validación y análisis.

Antes de continuar, se verifica la cantidad de registros obtenidos para controlar que la información se haya incorporado correctamente.

In [ ]:
cyclistic = pd.concat(datos_ajustados, ignore_index=True)

print("Registros totales:", len(cyclistic))
print("Columnas:", len(cyclistic.columns))

display(cyclistic.head())

Registros totales: 6115982
Columnas: 19


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length_min,ride_length_sec,ride_length_min_original,ajuste_horario,extremo,horario_ambiguo
0,B0AD82B8A2556E7C,electric_bike,2025-09-01 12:05:13.323,2025-09-01 12:25:19.273,NaN,NaN,NaN,NaN,41.91,-87.63,41.96,-87.64,member,20.099167,1205.950,NaN,0,0,0
1,DCC4ABF0F7DCCE6A,electric_bike,2025-09-01 01:41:00.775,2025-09-01 01:48:59.228,NaN,NaN,NaN,NaN,41.95,-87.67,41.95,-87.65,member,7.974217,478.453,NaN,0,0,0
2,C27785FFE7568433,electric_bike,2025-09-01 07:16:13.921,2025-09-01 07:39:51.831,NaN,NaN,NaN,NaN,41.94,-87.65,41.98,-87.65,member,23.631833,1417.910,NaN,0,0,0
3,812DB2D3AD2F18EB,electric_bike,2025-09-01 09:31:36.294,2025-09-01 09:49:08.022,NaN,NaN,NaN,NaN,41.98,-87.65,41.94,-87.65,member,17.528800,1051.728,NaN,0,0,0
4,61F6BD3174DC4E84,electric_bike,2025-09-26 08:51:58.069,2025-09-26 09:07:29.290,NaN,NaN,NaN,NaN,41.90,-87.69,41.87,-87.65,member,15.520350,931.221,NaN,0,0,0


## Identificación y tratamiento de registros duplicados

Una vez consolidados los archivos mensuales, se revisa la existencia de identificadores de viaje repetidos (`ride_id`).

Dado que cada identificador debería corresponder a un viaje único, se utiliza esta variable para detectar posibles duplicaciones.

Los registros duplicados se identifican y se eliminan conservando una única ocurrencia de cada identificador, evitando que un mismo viaje se contabilice más de una vez en los análisis posteriores.

In [ ]:
# Verificar identificadores duplicados
duplicados = cyclistic["ride_id"].duplicated().sum()

print("Registros totales:", len(cyclistic))
print("Ride ID únicos:", cyclistic["ride_id"].nunique())
print("Registros con ride_id duplicado:", duplicados)

Registros totales: 6115982
Ride ID únicos: 6115947
Registros con ride_id duplicado: 35


In [ ]:
# Seleccionar todos los registros con ride_id repetido
casos_duplicados = cyclistic[
    cyclistic["ride_id"].duplicated(keep=False)
].copy()

# Ordenarlos para facilitar la comparación
casos_duplicados = casos_duplicados.sort_values("ride_id")

print("Registros involucrados:", len(casos_duplicados))
print("Identificadores repetidos:", casos_duplicados["ride_id"].nunique())

# Ver si los registros originales son idénticos
columnas_originales = datos_mensuales[0].columns[:13].tolist()

duplicados_exactos = casos_duplicados.duplicated(
    subset=columnas_originales
).sum()

print("Duplicados exactos:", duplicados_exactos)

Registros involucrados: 70
Identificadores repetidos: 35
Duplicados exactos: 35


In [ ]:
# Conservar la primera aparición de cada ride_id
cyclistic_limpio = cyclistic.drop_duplicates(
    subset="ride_id",
    keep="first"
).copy()

# Comprobar el resultado
print("Registros originales:", len(cyclistic))
print("Registros eliminados:", len(cyclistic) - len(cyclistic_limpio))
print("Registros finales:", len(cyclistic_limpio))
print("Duplicados restantes:", cyclistic_limpio["ride_id"].duplicated().sum())

Registros originales: 6115982
Registros eliminados: 35
Registros finales: 6115947
Duplicados restantes: 0


In [ ]:
print("Registros finales:", len(cyclistic_limpio))

print("Ride ID duplicados:",
      cyclistic_limpio["ride_id"].duplicated().sum())

print("Duraciones no positivas:",
      (cyclistic_limpio["ride_length_min"] <= 0).sum())

print("Duraciones nulas:",
      cyclistic_limpio["ride_length_min"].isna().sum())

print("Membresías inválidas:",
      (~cyclistic_limpio["member_casual"].isin(["member", "casual"])).sum())

print("\nCorrecciones horarias:")
print(cyclistic_limpio["ajuste_horario"].value_counts())

print("\nViajes con horario ambiguo:",
      cyclistic_limpio["horario_ambiguo"].sum())

print("\nViajes extremos:",
      cyclistic_limpio["extremo"].sum())

Registros finales: 6115947
Ride ID duplicados: 0
Duraciones no positivas: 0
Duraciones nulas: 0
Membresías inválidas: 0

Correcciones horarias:
ajuste_horario
 0     6115890
 60         29
-60         28
Name: count, dtype: int64

Viajes con horario ambiguo: 362

Viajes extremos: 188908


In [ ]:
# Crear carpeta para los datos procesados
carpeta_procesados = "/content/drive/MyDrive/Colab Notebooks/cyclistic/datos_procesados"

os.makedirs(carpeta_procesados, exist_ok=True)

# Guardar la base final
ruta_salida = os.path.join(carpeta_procesados, "cyclistic_202509_202608_limpio.csv")

cyclistic_limpio.to_csv(ruta_salida, index=False)

print("Archivo guardado en:", ruta_salida)

Archivo guardado en: /content/drive/MyDrive/Colab Notebooks/cyclistic/datos_procesados/cyclistic_202509_202608_limpio.csv
